In [1]:
#1 ทำ preprocessing สำหรับ log
import re

def clean_log(text):
    # ลบ timestamp (optional แต่แนะนำ)
    text = re.sub(r"\d{4}-\d{2}-\d{2}T.*?Z", "", text)
    # แปลง tab เป็น space
    text = text.replace("\t", " ")
    return text.strip()

In [4]:
#2 โหลด CSV + clean

from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="test_set.csv"
)["train"]

dataset = dataset.map(
    lambda x: {"text": clean_log(x["query log"])}
)

dataset = dataset.remove_columns(["query log", "status"])
dataset = dataset.rename_column("label", "labels")


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/499500 [00:00<?, ? examples/s]

In [5]:
#3 ทำ Tokenization
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "google-bert/bert-base-uncased"
)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)


Map:   0%|          | 0/499500 [00:00<?, ? examples/s]

In [4]:
tokenizer("hello world")


{'input_ids': [101, 7592, 2088, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}

In [6]:
#4 Train / Validation Split

dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_ds = dataset["train"]
val_ds = dataset["test"]


In [6]:
#5 โหลดโมเดลสำหรับ Binary Classification

from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=2
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
#6 Training Configuration (เหมาะกับ Log)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="Finetuned Bert Model",
    eval_strategy="epoch", #เลิกใช้ evaluation_strategy แล้ว
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",   # 🔴 ปิด wandb
)


In [8]:
#7 Metric (สำคัญมากสำหรับ Anomaly)

from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [9]:
#8 เริ่ม Fine-tune 🚀

from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]  #หยุด Train เมื่อค่า F1 ไม่ดีขึ้น
)

trainer.train()


C:\Users\aungl\AppData\Local\Temp\ipykernel_1708\3697966789.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.324565,0.920000,0.947368,0.915254,0.931034
2,0.384900,0.108213,0.970000,0.982759,0.966102,0.974359
3,0.384900,0.035705,1.000000,1.000000,1.000000,1.000000
4,0.068000,0.019557,1.000000,1.000000,1.000000,1.000000


TrainOutput(global_step=100, training_loss=0.2264613151550293, metrics={'train_runtime': 25.8683, 'train_samples_per_second': 61.852, 'train_steps_per_second': 3.866, 'total_flos': 105244422144000.0, 'train_loss': 0.2264613151550293, 'epoch': 4.0})